# 02 - Análise de Confusão, Identidade de Pares, GradCAM, Entropia Hierárquica e Calibração Formal

Reproduz as análises de interpretabilidade e incerteza de short/analize_convnext.ipynb e short/full_loss_function.ipynb que sustentam as **Tabelas 4 e 7**:
- Extração de pares de classes mais confundidos via matriz de confusão e coleta de amostras.
- Alinhamento global pareado (identidade de sequência) entre espécies confundidas (offline-safe sem chamadas obrigatórias de rede ao NCBI).
- Atribuição 1D com GradCAM e ScoreCAM mascarando posições de padding.
- Análise de entropia hierárquica preditiva (H_species, H_genus, Delta H) e entropia normalizada de ativação pré-classificadora (H_latent / ln D) via TaxonomyEntropyAnalyzer (Tabela 7).
- Métricas formais de calibração probabilística (ECE 15 bins, MCE, Brier score, NLL, Reliability Diagram, Temperature Scaling e auditoria por ruído sob AD-013).


In [ ]:
import os
import sys
from pathlib import Path

# Resolução autônoma de caminho para JupyterHub remoto e desenvolvimento local
current_dir = Path.cwd().resolve()
for r in [current_dir, current_dir.parent, current_dir.parent.parent, current_dir.parent.parent.parent]:
    if (r / "src" / "viral_cnn").exists():
        if str(r / "src") not in sys.path:
            sys.path.insert(0, str(r / "src"))
        break

import torch
import pandas as pd
import numpy as np
import json
from datetime import datetime, timezone

from viral_cnn.notebook_runtime import (
    setup_notebook_runtime,
    ensure_smoke_test_artifacts,
    write_notebook_manifest,
    resolve_split_files,
    create_optimized_dataloader,
    free_gpu_memory,
)
from viral_cnn.models.registry import build_registered_model
from viral_cnn.data.parquet_dataset import ParquetBatchIterableDataset
from viral_cnn.taxonomy.mapping import load_taxonomy_mapping, model_label_to_taxonomic_id, build_species_to_genus_tensor
from viral_cnn.eval.confusion import (
    compute_and_plot_confusion_insights,
    get_confused_class_samples,
    compare_confused_species_identity,
)
from viral_cnn.interpretability.gradcam import GradCAM1D, ScoreCAM1D
from viral_cnn.eval.calibration import (
    TaxonomyEntropyAnalyzer,
    compute_calibration_metrics,
    compute_reliability_diagram,
    TemperatureScaler,
    audit_model_calibration_by_noise,
    audit_dataloader_calibration_by_noise,
    save_reliability_comparison,
)

# 1. Initialize Runtime and Environment (AD-017)
# Por padrão (REMOTE_SERVER != False), conecta automaticamente à topologia do servidor remoto (/home/mlima/workspace)
# e desativa o modo smoke test (SMOKE_TEST=0).
# Para rodar em modo local de teste sintético rápido em CPU, defina: os.environ['REMOTE_SERVER'] = 'False'
from viral_cnn.m02_workflow import begin_m02
run = begin_m02("short/02")
ctx = run.ctx
output_dir = run.output_dir
c04_run_id = run.run_id
print(f"Device: {ctx.device}")
print(f"Smoke Test Mode: {ctx.is_smoke_test}")
print(f"Data Root: {ctx.data_root}")


In [ ]:
with run.guard():
    # 2. Setup Artifacts and Data Loaders
    if ctx.is_smoke_test:
        run.prepare_smoke()



    species_parquet_dir = Path(ctx.topology.paths["short"]["species_parquet_dir"])
    species_test_files = run.split_files("species", "test")
    test_ds_species = ParquetBatchIterableDataset(
        species_test_files,
        batch_size=32 if ctx.is_smoke_test else 4096,
        shuffle=False,
        split="test",
        include_identity=True,
    )
    test_loader = create_optimized_dataloader(
        test_ds_species,
        is_smoke_test=ctx.is_smoke_test,
        device=ctx.device,
    )

    species_val_files = run.split_files("species", "val")
    val_loader = None
    if species_val_files:
        val_ds_species = ParquetBatchIterableDataset(
            species_val_files,
            batch_size=32 if ctx.is_smoke_test else 4096,
            shuffle=False,
            split="val",
            include_identity=True,
        )
        val_loader = create_optimized_dataloader(
            val_ds_species,
            is_smoke_test=ctx.is_smoke_test,
            device=ctx.device,
        )

    mapping_csv = ctx.topology.paths["short"]["species_csv"]
    species_mapping = {}
    if os.path.exists(mapping_csv):
        species_mapping = load_taxonomy_mapping(mapping_csv, rank_column="species")
    print(f"Taxonomy mapping loaded with {len(species_mapping)} taxa.")


In [ ]:
with run.guard():
    # 3. Confusion Matrix Insights and Top-K Confused Pairs
    eval_model_name = "resnet_clean_species"
    ckpt = ctx.topology.paths["checkpoints"].get(eval_model_name) if not ctx.is_smoke_test else None
    if not ctx.is_smoke_test and (ckpt is None or not Path(ckpt).exists()):
        raise FileNotFoundError(f"Checkpoint for {eval_model_name} not found: {ckpt}")
    model = build_registered_model(eval_model_name, checkpoint_path=ckpt, device=ctx.device)
    num_classes = 5 if ctx.is_smoke_test else 13683

    top_k_confused = compute_and_plot_confusion_insights(
        model=model,
        test_loader=test_loader,
        device=ctx.device,
        num_classes=num_classes,
        top_k_confusion=5,
        mapping_csv_path=mapping_csv,
        show=False,
    )
    print(f"Top confused pairs: {len(top_k_confused)}")

    # Collect representative samples of confused classes
    pairs_to_collect = top_k_confused if top_k_confused else [(0.5, 0, 1)]
    confused_samples = get_confused_class_samples(
        confused_pairs_info=pairs_to_collect,
        test_loader=test_loader,
        device=ctx.device,
        n_examples_per_class=2,
        max_batches=2 if ctx.is_smoke_test else 50,
    )
    print(f"Collected samples for {len(confused_samples)} target classes.")


In [ ]:
with run.guard():
    # 4. Pairwise Sequence Identity between Confused Species (Offline-Safe)
    samples_by_id = {item["class_true"]["id"]: item["class_true"]["examples"] for item in confused_samples}

    identity_records = []
    pairs_to_compare = top_k_confused[:3] if top_k_confused else [(0.5, 0, 1)]
    for score, true_lbl, pred_lbl in pairs_to_compare:
        true_exs = samples_by_id.get(true_lbl, [])
        pred_exs = samples_by_id.get(pred_lbl, [])

        if not true_exs or not pred_exs:
            if not ctx.is_smoke_test:
                raise RuntimeError(
                    f"Could not retrieve real batch samples for confused pair (true: {true_lbl}, pred: {pred_lbl}). "
                    "Ensure test_loader yields both classes within max_batches."
                )
            seq1 = true_exs[0] if true_exs else torch.randint(0, 4, (512,))
            seq2 = pred_exs[0] if pred_exs else torch.randint(0, 4, (512,))
        else:
            seq1 = true_exs[0]
            seq2 = pred_exs[0]

        id_res = compare_confused_species_identity(
            seq1=seq1,
            seq2=seq2,
            label1=species_mapping.get(true_lbl, f"Taxon_{true_lbl}"),
            label2=species_mapping.get(pred_lbl, f"Taxon_{pred_lbl}"),
            use_ncbi_online=False,
        )
        identity_records.append({
            "confusion_score": round(score, 4),
            "true_label": true_lbl,
            "pred_label": pred_lbl,
            "species_true": species_mapping.get(true_lbl, f"Taxon_{true_lbl}"),
            "species_pred": species_mapping.get(pred_lbl, f"Taxon_{pred_lbl}"),
            "identity_percent": id_res["identity"],
            "alignment_score": id_res["alignment_score"],
        })

    df_identity = pd.DataFrame(identity_records)
    df_identity.to_csv(output_dir / "confused_pairs_identity.csv", index=False)
    print("Pairwise sequence identity table saved:")
    print(df_identity.to_string())
    try:
        display(df_identity)
    except NameError:
        pass


In [ ]:
with run.guard():
    # 5. 1D GradCAM / ScoreCAM Interpretability Attribution
    if ctx.is_smoke_test:
        cam_explainer = GradCAM1D(model=model, target_layer=model.layer4)
        sample_input = torch.randint(0, 4, (1, 512), dtype=torch.long, device=ctx.device)
        cam_map = cam_explainer(sample_input, target_class=0)
        print(f"GradCAM 1D attribution shape: {cam_map.shape}, min: {cam_map.min():.4f}, max: {cam_map.max():.4f}")
        gradcam_status = "synthetic_demonstration"
    else:
        gradcam_status = "omitted_empirical"
        print("GradCAM random demonstration omitted in empirical mode.")
    (output_dir / "gradcam_status.json").write_text(json.dumps({"status": gradcam_status, "biological_evidence": False}))
    free_gpu_memory(model)
    del model


In [ ]:
with run.guard():
    # 6. Hierarchical Predictive Entropy & Pre-Classifier Activation (Table 7)
    # Build species-to-genus mapping tensor
    if ctx.is_smoke_test:
        species_to_genus = torch.tensor([0, 0, 1, 1, 2], dtype=torch.long, device=ctx.device)
        tax_path = None
    else:
        tax_path = Path(ctx.topology.paths["short"]["hybrid_taxonomy_parquet"])
        if tax_path.is_dir():
            candidates = [tax_path / n for n in ("main_taxonomy_hybrid_CLEANSED.parquet", "main_taxonomy_hybrid_ictv_ncbi.parquet")]
            tax_path = next((p for p in candidates if p.is_file()), candidates[0])
        if not tax_path.is_file():
            raise FileNotFoundError(f"Taxonomy parquet not found: {tax_path}")
        species_to_genus = build_species_to_genus_tensor(tax_path, num_species=13683, device=ctx.device)

    species_model_variants = (
        ["resnet_clean_species", "convnext_zero_species"] if ctx.is_smoke_test
        else [
            "resnet_clean_species",
            "resnet_noise_species",
            "convnext_zero_species",
            "convnext_transfer_species",
            "convnext_noise_species",
        ]
    )

    entropy_records = []
    for model_name in species_model_variants:
        ckpt = ctx.topology.paths["checkpoints"].get(model_name) if not ctx.is_smoke_test else None
        if not ctx.is_smoke_test and (ckpt is None or not Path(ckpt).exists()):
            raise FileNotFoundError(f"Checkpoint for {model_name} not found: {ckpt}")
        m = build_registered_model(model_name, checkpoint_path=ckpt, device=ctx.device)
        with TaxonomyEntropyAnalyzer(m, species_to_genus, device=ctx.device) as analyzer:
            ent_summary = analyzer.analyze_dataloader(test_loader, max_batches=2 if ctx.is_smoke_test else 20)
            ent_summary["model_name"] = model_name
            entropy_records.append(ent_summary)
            print(f"{model_name} -> H_species: {ent_summary['mean_h_species']:.4f}, H_genus: {ent_summary['mean_h_genus']:.4f}, Gap: {ent_summary['mean_hierarchical_gap']:.4f}, H_act: {ent_summary['mean_h_activation']:.4f}")
        free_gpu_memory(m)
        del m

    df_entropy = pd.DataFrame(entropy_records)
    df_entropy.to_csv(output_dir / "table7_hierarchical_entropy.csv", index=False)
    print("Table 7 Hierarchical entropy saved:")
    print(df_entropy.to_string())
    try:
        display(df_entropy)
    except NameError:
        pass


In [ ]:
with run.guard():
    # 7. Formal Calibration Auditing across Controlled Noise Levels (AD-013)
    from tqdm.auto import tqdm

    cal_model_variants = (
        ["resnet_clean_species", "convnext_zero_species"] if ctx.is_smoke_test
        else [
            "resnet_clean_species",
            "resnet_noise_species",
            "convnext_zero_species",
            "convnext_transfer_species",
            "convnext_noise_species",
        ]
    )

    noise_rates = [0.0, 0.05] if ctx.is_smoke_test else [0.0, 0.02, 0.05, 0.10, 0.15]
    max_batches = 2 if ctx.is_smoke_test else None
    val_max_batches = 2 if ctx.is_smoke_test else None
    total_test_batches = max_batches if max_batches is not None else len(test_loader)

    cal_rows = []
    for m_name in tqdm(cal_model_variants, desc="Calibration Models"):
        ckpt = ctx.topology.paths["checkpoints"].get(m_name) if not ctx.is_smoke_test else None
        if not ctx.is_smoke_test and (ckpt is None or not Path(ckpt).exists()):
            raise FileNotFoundError(f"Checkpoint for {m_name} not found: {ckpt}")
        m_eval = build_registered_model(m_name, checkpoint_path=ckpt, device=ctx.device)
        cal_report = audit_dataloader_calibration_by_noise(
            models_dict={m_name: m_eval},
            test_loader=tqdm(
                test_loader,
                total=total_test_batches,
                desc=f"{m_name} (test)",
                leave=False,
            ),
            val_loader=val_loader,
            noise_rates=noise_rates,
            device=ctx.device,
            n_bins=15,
            max_batches=max_batches,
            val_max_batches=val_max_batches,
            require_provenance=True,
        )
        for n_rate, m_metrics in cal_report["models"][m_name].items():
            if not n_rate.startswith("noise_"):
                continue
            row = {"model": m_name, "noise": n_rate}
            row.update(m_metrics)
            row["brier"] = row["brier_score"]
            row["reliability_bins"] = m_metrics.get("reliability_bins", {})
            row["temperature_state"] = m_metrics.get("temperature_state", "UNAVAILABLE")
            row["temperature_provenance"] = cal_report["models"][m_name].get("temperature")
            cal_rows.append(row)
        free_gpu_memory(m_eval)
        del m_eval

    df_cal = pd.DataFrame(cal_rows)
    df_cal.drop(columns=["reliability_bins"], errors="ignore").to_csv(output_dir / "calibration_audit_noise.csv", index=False)
    figure_records = []
    for row in cal_rows:
        bins = row.get("reliability_bins", {})
        if "before" not in bins or "after" not in bins:
            continue
        fig_paths = save_reliability_comparison(
            bins["before"], bins["after"], output_dir / "reliability",
            title=f"{row['model']} | {row['noise']} | T={row.get('temperature_provenance', {}).get('temperature', float('nan')):.4g} | N={row.get('n_samples', 0)}",
            filename_prefix=f"{row['model']}_{row['noise']}",
            metadata={"model": row["model"], "noise": row["noise"], "temperature": row.get("temperature_provenance", {}).get("temperature"), "n": row.get("n_samples"), "seed": 42, "is_synthetic": ctx.is_smoke_test},
        )
        figure_records.append({"model": row["model"], "noise": row["noise"], **fig_paths})
    (output_dir / "c04_reliability_figures.json").write_text(json.dumps(figure_records, indent=2))
    figure_by_cell = {(r["model"], r["noise"]): r for r in figure_records}
    cal_rows_by_cell = {(r["model"], r["noise"]): r for r in cal_rows}
    expected_cal_cells = [(m, f"noise_{rate}") for m in cal_model_variants for rate in noise_rates]
    cal_cell_status = []
    for model_name, noise_name in expected_cal_cells:
        figure = figure_by_cell.get((model_name, noise_name), {})
        row = cal_rows_by_cell.get((model_name, noise_name), {})
        ok = bool(figure.get("png") and figure.get("pdf") and row.get("temperature_state") == "SEALED")
        reason = None if ok else (figure.get("error") or row.get("temperature_provenance", {}).get("reason") or "condition/temperature/figure unavailable")
        cal_cell_status.append({"model": model_name, "noise": noise_name, "status": "complete" if ok else "unavailable", "reason": reason})
    (output_dir / "c04_calibration_completeness.json").write_text(json.dumps({"expected": len(expected_cal_cells), "complete": sum(r["status"] == "complete" for r in cal_cell_status), "cells": cal_cell_status, "figures": figure_records}, indent=2))
    print("Calibration audit table saved:")
    print(df_cal.to_string())
    try:
        display(df_cal)
    except NameError:
        pass


In [ ]:
with run.guard():
    # 8. Write Execution Manifest
    manifest = write_notebook_manifest(
        notebook_id="02_analyze_confidence_and_calibration",
        data_root=ctx.data_root,
        is_smoke_test=ctx.is_smoke_test,
        input_files=[*species_test_files, *species_val_files],
        metrics_summary={"top_k_confused": len(top_k_confused), "c04_calibration_protocol": "c04_clean_validation_temperature_v1", "reliability_panels": len(figure_records), "expected_panels": len(cal_model_variants) * len(noise_rates)},
        run_id=c04_run_id,
        configuration={"n_bins": 15, "binning": "equal_width", "validation_max_batches": val_max_batches},
        outputs=[output_dir / "c04_reliability_figures.json", output_dir / "c04_calibration_completeness.json", output_dir / "calibration_audit_noise.csv", *[Path(f[k]) for f in figure_records for k in ("csv", "json", "png", "pdf") if f.get(k)]],
        output_json_path=output_dir / "c04_notebook_manifest.json",
    )
    print("Manifest successfully written.")

    if not all(cell["status"] == "complete" for cell in cal_cell_status):
        raise RuntimeError("Incomplete C04 calibration/figures")
    used_checkpoints = {n: str(ctx.topology.paths["checkpoints"][n]) for n in set(cal_model_variants + species_model_variants + [eval_model_name])} if not ctx.is_smoke_test else {}
    manifest = run.complete(input_files=[*species_test_files, *species_val_files, mapping_csv] + ([tax_path] if tax_path else []), checkpoints=used_checkpoints)
